# Gas Consumption Forecasting

## Project Overview

Forecasts **daily natural gas consumption** (GWh) over a 14-day horizon. Synthetic data spans 2 years with strong winter heating peaks, weekly industrial patterns, and price sensitivity.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily gas consumption, predict the next 14 days. Gas utilities and pipeline operators need demand forecasts for supply scheduling, storage management, and procurement.

## Why This Project Matters

Natural gas is the primary heating fuel in many regions. Demand swings dramatically with temperature. Under-forecasting leads to supply shortages and price spikes; over-forecasting wastes storage capacity and procurement costs.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily gas consumption
- Strong winter peak (heating demand)
- Weekly pattern (industrial load weekdays)
- Summer baseline (cooking, hot water, industrial)
- Gradual efficiency trend (slightly declining)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `gas_gwh` | Daily gas consumption (GWh) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'gas_gwh'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 120 - 0.01 * t  # slight efficiency improvement
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 8
    else: weekly[i] = -12

# Strong winter peak (heating)
seasonal = 80 * np.cos(2 * np.pi * (t - 15) / 365)

# Holiday dips
holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -25

noise = np.random.normal(0, 8, N_POINTS)
gas_gwh = base + weekly + seasonal + holiday + noise
gas_gwh = np.maximum(gas_gwh, 20).round(1)

df = pd.DataFrame({'date': dates, 'gas_gwh': gas_gwh})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,gas_gwh
0,2022-01-01,164.3
1,2022-01-02,159.6
2,2022-01-03,211.2
3,2022-01-04,218.5
4,2022-01-05,204.7


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('gas_gwh Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("gas_gwh Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

gas_gwh Statistics:
count    730.000000
mean     117.861507
std       57.054819
min       20.000000
25%       64.325000
50%      119.050000
75%      170.600000
max      221.100000
Name: gas_gwh, dtype: float64

CV: 0.484
Skewness: 0.003


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       16.3 | RMSE:       18.9 | MAPE:  9.49%
Seasonal Naive (7)             | MAE:       14.0 | RMSE:       17.1 | MAPE:  8.54%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                            
KernelRidge                        1129.770447 -85.828496  114.012871    0.037310
GaussianProcessRegressor            638.600968 -48.046228   85.689094    0.274342
QuantileRegressor                   430.511515 -32.039347   70.329652    0.067373
DummyRegressor                      398.461534 -29.573964   67.654800    0.014129
LarsCV                               97.455664  -6.419666   33.328436    0.033160
NuSVR                                71.281950  -4.406304   28.449390    0.035049
SVR                                  44.119460  -2.316882   22.283722    0.032300
MLPRegressor                         37.148691  -1.780669   20.403156    0.479243
DecisionTreeRegressor                27.630311  -1.048485   17.512139    0.019980
PassiveAggressiveRegressor           27.047848  -1.003681   17.319565    0.01

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:       10.0 | RMSE:       13.1 | MAPE:  5.60%
FLAML Test (lgbm)              | MAE:        9.3 | RMSE:       11.2 | MAPE:  5.73%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       22.6 | RMSE:       27.6 | MAPE: 14.13%
SF AutoETS                     | MAE:       21.4 | RMSE:       26.3 | MAPE: 13.40%
SF SeasonalNaive               | MAE:       15.5 | RMSE:       18.9 | MAPE:  9.70%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model       MAE      RMSE      MAPE
 FLAML Test (lgbm)  9.288940 11.202978  5.731136
      FLAML (lgbm) 10.027131 13.055630  5.600546
Seasonal Naive (7) 13.950000 17.054807  8.543342
  SF SeasonalNaive 15.542857 18.890738  9.703658
Naive (Last Value) 16.285714 18.941979  9.487642
        SF AutoETS 21.433784 26.271821 13.401915
      SF AutoARIMA 22.588170 27.600322 14.129492

Top 3:
             model       MAE      RMSE     MAPE
 FLAML Test (lgbm)  9.288940 11.202978 5.731136
      FLAML (lgbm) 10.027131 13.055630 5.600546
Seasonal Naive (7) 13.950000 17.054807 8.543342


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -4.22, Std: 10.38


## Interpretation and Business Insight

- Gas consumption has the strongest seasonal pattern of utility commodities
- Winter heating drives demand 3-4x above summer baseline
- Industrial load creates consistent weekday patterns
- Efficiency improvements create a slow downward trend
- Temperature (HDD — heating degree days) is the key missing feature

## Limitations

1. Synthetic — real gas demand is highly temperature-sensitive
2. No temperature/HDD features
3. No gas price signal
4. No sector decomposition (residential vs industrial vs power gen)
5. Daily granularity misses intraday peaks

## How to Improve This Project

1. Add heating degree days (HDD) as primary feature
2. Include gas spot prices for price-responsive demand
3. Segment by customer class
4. Use weather ensemble forecasts for probabilistic demand
5. Model cold snaps separately (extreme events)

## Production Considerations

- Day-ahead and week-ahead forecasting for pipeline operations
- Integration with weather forecast APIs
- Storage injection/withdrawal optimization
- Supply procurement planning

## Common Mistakes

1. Ignoring temperature — HDD explains 80%+ of winter variance
2. Using summer models for winter (regime change)
3. Not handling cold snap events separately
4. Ignoring gas-electric switching effects

## Mini Challenge / Exercises

1. Add synthetic HDD (heating degree days) and measure improvement
2. Model winter and summer separately — compare fit
3. Detect anomalous demand days (cold snaps)
4. Try weekly aggregation for procurement planning

## Final Summary / Key Takeaways

1. Gas consumption has extreme seasonality (winter >> summer)
2. Temperature/HDD is the single most important predictor
3. Weekly industrial patterns add a second cycle
4. Efficiency trends create slow structural decline
5. Probabilistic forecasts are essential for storage optimization

In [18]:
import json
metrics = {
    'project': 'Gas Consumption Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Gas Consumption Forecasting — Complete ===")

Metrics saved.

=== Gas Consumption Forecasting — Complete ===
